# A1: MFCC + GMM-HMM (Kaldi, CPU)

Runs in a **CPU** Colab runtime (High-RAM if available), in a second session next to the C1 training.
Kaldi comes from conda-forge (no compiling); its `steps/` and `utils/` scripts from the Kaldi GitHub repo.
Stages: data + grapheme lexicon, lang, 3-gram LM, MFCC + CMVN, mono, tri1, tri2b (LDA+MLLT), tri3b (SAT), decode, LM weight on dev.

Settings are chosen on **dev** only; **test** once, after the freeze.

In [ ]:
!nproc; free -g | head -2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nlp3_gmm_hmm'
!mkdir -p $OUT

In [ ]:
REPO = 'https://github.com/ata-sesli/nlp-project-3'
BRANCH = 'main'
!git clone --depth 1 --filter=blob:none --sparse -b $BRANCH $REPO /content/nlp-project-3
%cd /content/nlp-project-3
!git sparse-checkout set data/clean data/raw/usc/train data/raw/usc/dev baselines whisper_finetune benchmark
!ls data/clean baselines/gmm_hmm

In [ ]:
# Kaldi from conda-forge in its own environment (~3 min)
%env MAMBA_ROOT_PREFIX=/content/mamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
!micromamba create -y -q -n kaldi -c conda-forge kaldi=5.5.1172 sox libflac pysoundfile=0.14.0 numpy python=3.12
!micromamba run -n kaldi python -m pip install -q jiwer==4.0.0
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/kaldi-asr/kaldi /content/kaldi-src
!cd /content/kaldi-src && git sparse-checkout set egs/wsj/s5/steps egs/wsj/s5/utils && git rev-parse HEAD
!micromamba run -n kaldi which compute-mfcc-feats gmm-init-mono arpa2fst flac

In [ ]:
import os
NJ = os.cpu_count()
ARPA = ''  # Elif's ARPA (lm/...) once ready; empty = our lm_arpa.py 3-gram
RUN = f'micromamba run -n kaldi bash baselines/gmm_hmm/run.sh --work /content/gmm_hmm_work --kaldi-src /content/kaldi-src --nj {NJ}'
if ARPA:
    RUN += f' --arpa {ARPA}'
# stages 0-3: data dirs, lexicon, lang, LM, MFCC
!{RUN} --stage 0 --stop-stage 3 2>&1 | tee $OUT/stages_0-3.log | tail -30

In [ ]:
# stages 4-7: mono, tri1, tri2b, tri3b (the long part). If time is short, use --stop-stage 5 and --model tri1 below.
!{RUN} --stage 4 --stop-stage 7 2>&1 | tee $OUT/stages_4-7.log | tail -30
!tar -czf $OUT/exp_models.tgz -C /content/gmm_hmm_work exp/tri1 exp/tri2b exp/tri3b data/lang data/lang_test data/local/dict 2>/dev/null; ls -lh $OUT

In [ ]:
# stages 8-9: graph, dev decode, LM weight + insertion penalty on dev, predictions + run log
!{RUN} --stage 8 --model tri3b --splits dev 2>&1 | tee $OUT/stages_8-9_dev.log | tail -45
!cp -r benchmark $OUT/

## Test: once, after the freeze (Sat 21 Nov)

In [ ]:
!git sparse-checkout add data/raw/usc/test
!{RUN} --eval-only true --model tri3b --splits test --after-freeze true 2>&1 | tee $OUT/test.log | tail -10
!cp -r benchmark $OUT/